# ICT-48 — Onset/SAE Qwen3.5-9B-Base, agregat 4 graines (W et S)

> **Statut epistemique** -- **Etabli** : sur 4 graines (seeds 0,1,2,3) du finetune LoRA 4-bit Qwen/Qwen3.5-9B-Base (lr=1e-05, 120 steps, num_generations=4), l'onset statique se maintient pour les deux bras W (world) et S (spurious). Aucune transition vers le takeover, ni inversion de signe. Cross-architecture robustness avec ICT-47 9B (c.228 cos 0.60 min 0.46).

> **Sources** : artifacts `D:/Runs/ICT-onset-9b-c233/onset_qwen35_9b_base.json` (seeds 0,1, c.247-quater) et `D:/Runs/ICT-onset-9b-c246/onset_qwen35_9b_base_seeds23.json` (seeds 2,3, c.246). Total 8 runs : 4 W + 4 S.

## Contexte

Les runs c.233 et c.246 ont ete conduits par le **regisseur GPU de la flotte** (lane `myia-ai-01:CoursIA-2`, mandat 2026-10-05) sur le GPU 2 de la machine ai-01. Les deux jobs ont ete par rapport a une **reservation hors-ledger** (le ledger `gpu-reservation` n'a ete active qu'a partir de c.257, 2026-10-08T03:25Z). Les artefacts sont preserves dans `D:/Runs/ICT-onset-9b-c*/`.

Le present notebook agrege les 4 graines (W seed 0,1,2,3 + S seed 0,1,2,3) en statistiques descriptives et trace la mediane par step pour chaque bras.

## Garde-fous d'honnetete (a lire avant les resultats)

1. **Pas de SLI** : les runs utilisent `num_generations=4` (rollout group size), pas un SFT classique. Le signal est un **hack rate** sur 24 prompts fixes x 4 generations, et un **mc rate** (majority-choice correctness) sur le meme panel.
2. **Statique vs takeover** : on observe `mc_late - mc_early` (Dmc) et `hack_late - hack_early` (Dhack). Une transition vers le takeover se manifesterait par un Dmc positif ET un Dhack negatif (le modele apprend a court-circuiter le signal de recompense). Une stagnation des deux signe onset statique.
3. **Cross-architecture** : le pilote ICT-47 9B (c.228) et le present 9B Qwen3.5-Base maintiennent l'onset statique. Pas un artefact d'un modele.
4. **Mesures** : 4 graines, lr=1e-05, max_steps=120, 4-bit QLoRA env `coursia-sae`. Total ~2h30 par job (2 bras x 2 seeds).

## Protocole d'agregation

1. Charger les deux artefacts JSON (c.233 = seeds 0,1, c.246 = seeds 2,3).
2. Concatenation par bras (W et S), obtenir 4 runs par bras.
3. Calculer mean et std pour `hack_early/late`, `mc_early/late`, `reward_early/late`.
4. Calculer la mediane par step (120 steps) sur les `per_step` (24 prompts x 4 generations) -> 480 valeurs par seed, agregees en 120 medianes par bras.
5. Tracer les courbes medianes W et S superposees, plus les bandes +-1 sigma.
6. Tester Dmc et Dhack par signe (bootstrap leger sur les 4 graines).

In [1]:
import json
import os
import statistics
from pathlib import Path

import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

RUNS = {
    "c.233": Path("D:/Runs/ICT-onset-9b-c233/onset_qwen35_9b_base.json"),
    "c.246": Path("D:/Runs/ICT-onset-9b-c246/onset_qwen35_9b_base_seeds23.json"),
}

artifacts = {tag: json.loads(p.read_text(encoding="utf-8")) for tag, p in RUNS.items()}
for tag, art in artifacts.items():
    print(f"{tag}: model={art['model']!r} seeds={art['seeds']} max_steps={art['max_steps']} lr={art['lr']}")

c.233: model='Qwen/Qwen3.5-9B-Base' seeds=[0, 1] max_steps=120 lr=1e-05
c.246: model='Qwen/Qwen3.5-9B-Base' seeds=[2, 3] max_steps=120 lr=1e-05


In [2]:
# Concatener W et S sur les 4 graines
runs_w = artifacts["c.233"]["results"]["W"] + artifacts["c.246"]["results"]["W"]
runs_s = artifacts["c.233"]["results"]["S"] + artifacts["c.246"]["results"]["S"]
assert len(runs_w) == 4 and len(runs_s) == 4, f"Attendu 4 W + 4 S, got {len(runs_w)} W, {len(runs_s)} S"

print(f"Agregat : 4 W runs (seeds {sorted(r['seed'] for r in runs_w)}) + 4 S runs (seeds {sorted(r['seed'] for r in runs_s)})")
print(f"Total = 8 runs sur Qwen3.5-9B-Base.")

Agregat : 4 W runs (seeds [0, 1, 2, 3]) + 4 S runs (seeds [0, 1, 2, 3])
Total = 8 runs sur Qwen3.5-9B-Base.


In [3]:
def stats_for_arm(arm_runs, arm_label):
    keys = ["hack_early", "hack_late", "mc_early", "mc_late", "reward_early", "reward_late"]
    out = {"arm": arm_label, "n": len(arm_runs), "seeds": sorted(r["seed"] for r in arm_runs)}
    for k in keys:
        vals = [r[k] for r in arm_runs]
        out[k] = {"mean": round(statistics.mean(vals), 4), "std": round(statistics.stdev(vals), 4) if len(vals) > 1 else 0.0}
    return out

agg_w = stats_for_arm(runs_w, "W")
agg_s = stats_for_arm(runs_s, "S")

for agg in (agg_w, agg_s):
    print(f"\nBras {agg['arm']} (n={agg['n']}, seeds={agg['seeds']}):")
    for k in ("hack_early", "hack_late", "mc_early", "mc_late", "reward_early", "reward_late"):
        s = agg[k]
        print(f"  {k:14s}  mean={s['mean']:.4f}  std={s['std']:.4f}")

# Dhack et Dmc (changement entre early et late)
for agg in (agg_w, agg_s):
    d_hack = agg["hack_late"]["mean"] - agg["hack_early"]["mean"]
    d_mc = agg["mc_late"]["mean"] - agg["mc_early"]["mean"]
    print(f"\nBras {agg['arm']} : Dhack = {d_hack:+.4f} (early -> late), Dmc = {d_mc:+.4f}")


Bras W (n=4, seeds=[0, 1, 2, 3]):
  hack_early      mean=0.1781  std=0.0312
  hack_late       mean=0.2094  std=0.0208
  mc_early        mean=0.1031  std=0.0104
  mc_late         mean=0.1052  std=0.0138
  reward_early    mean=0.4135  std=0.0591
  reward_late     mean=0.4698  std=0.0307

Bras S (n=4, seeds=[0, 1, 2, 3]):
  hack_early      mean=0.2010  std=0.0168
  hack_late       mean=0.2062  std=0.0294
  mc_early        mean=0.4948  std=0.0334
  mc_late         mean=0.5156  std=0.0086
  reward_early    mean=0.8542  std=0.0335
  reward_late     mean=0.8990  std=0.0505

Bras W : Dhack = +0.0313 (early -> late), Dmc = +0.0021

Bras S : Dhack = +0.0052 (early -> late), Dmc = +0.0208


In [4]:
# Mediane par step sur les 4 graines
def median_per_step(arm_runs):
    n_steps = len(arm_runs[0]["per_step"])
    return [statistics.median([s[i] for s in (r["per_step"] for r in arm_runs)]) for i in range(n_steps)]

def mean_per_step(arm_runs):
    n_steps = len(arm_runs[0]["per_step"])
    return [statistics.mean([s[i] for s in (r["per_step"] for r in arm_runs)]) for i in range(n_steps)]

median_w = median_per_step(runs_w)
median_s = median_per_step(runs_s)
mean_w = mean_per_step(runs_w)
mean_s = mean_per_step(runs_s)
n_steps = len(median_w)
print(f"120 steps medianes calculees pour W et S.")

# Detection onset (premier step >= threshold)
def find_onset(seq, threshold=0.25):
    for i, v in enumerate(seq):
        if v >= threshold:
            return i, v
    return None, None

w_onset_idx, w_onset_val = find_onset(median_w)
s_onset_idx, s_onset_val = find_onset(median_s)
print(f"\nOnset (median, seuil 0.25) :")
print(f"  W : step {w_onset_idx} (median={w_onset_val})")
print(f"  S : step {s_onset_idx} (median={s_onset_val})")

# Mediane finale (step 119)
print(f"\nMediane finale (step 119) : W={median_w[-1]:.3f}  S={median_s[-1]:.3f}")
print(f"Mean finale   (step 119) : W={mean_w[-1]:.3f}  S={mean_s[-1]:.3f}")

120 steps medianes calculees pour W et S.

Onset (median, seuil 0.25) :
  W : step 0 (median=0.25)
  S : step 0 (median=0.375)

Mediane finale (step 119) : W=0.000  S=0.125
Mean finale   (step 119) : W=0.062  S=0.125


In [5]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), sharey=True)

# Panneau gauche : mediane par step, W et S
ax = axes[0]
xs = list(range(n_steps))
ax.plot(xs, median_w, label="W (world) -- mediane 4 graines", color="#1f77b4", lw=2)
ax.plot(xs, median_s, label="S (spurious) -- mediane 4 graines", color="#d62728", lw=2)
if w_onset_idx is not None:
    ax.axvline(w_onset_idx, color="#1f77b4", ls=":", alpha=0.6, label=f"W onset step {w_onset_idx}")
if s_onset_idx is not None:
    ax.axvline(s_onset_idx, color="#d62728", ls=":", alpha=0.6, label=f"S onset step {s_onset_idx}")
ax.set_xlabel("Step")
ax.set_ylabel("Mediane (24 prompts x 4 generations)")
ax.set_title("ICT-48 -- onset 9B Qwen3.5-Base, agregat 4 graines")
ax.legend(loc="upper left", fontsize=9)
ax.grid(alpha=0.3)

# Panneau droit : mean +- 1 sigma pour les 4 metriques early/late
ax = axes[1]
metrics = ["hack_early", "hack_late", "mc_early", "mc_late", "reward_early", "reward_late"]
x_pos = np.arange(len(metrics))
w_means = [agg_w[m]["mean"] for m in metrics]
w_stds = [agg_w[m]["std"] for m in metrics]
s_means = [agg_s[m]["mean"] for m in metrics]
s_stds = [agg_s[m]["std"] for m in metrics]
width = 0.35
ax.bar(x_pos - width/2, w_means, width, yerr=w_stds, label="W (world)", color="#1f77b4", capsize=3)
ax.bar(x_pos + width/2, s_means, width, yerr=s_stds, label="S (spurious)", color="#d62728", capsize=3)
ax.set_xticks(x_pos)
ax.set_xticklabels([m.replace("_", "\n") for m in metrics], fontsize=8)
ax.set_ylabel("Taux (mean +- 1 sigma sur 4 graines)")
ax.set_title("Metriques early/late -- agregat 4 graines")
ax.legend()
ax.grid(alpha=0.3, axis="y")

plt.tight_layout()
out_fig = Path("D:/CoursIA-2/MyIA.AI.Notebooks/IIT/ICT-Series/ict/results/ict48_onset_aggregate.png")
out_fig.parent.mkdir(parents=True, exist_ok=True)
plt.savefig(out_fig, dpi=110)
plt.show()
print(f"Figure sauvegardee : {out_fig}")

Figure sauvegardee : D:\CoursIA-2\MyIA.AI.Notebooks\IIT\ICT-Series\ict\results\ict48_onset_aggregate.png


<USER_PATH>\AppData\Local\Temp\ipykernel_<pid>\2687721047.py:40: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [6]:
# Test de signe bootstrap : Dhack et Dmc sur 4 graines
# H0 : la mediane des D par seed est nulle (onset statique)
# H1 : mediane > 0 (takeover) ou < 0 (regression)

import random

def bootstrap_sign_pvalue(per_seed_deltas, n_boot=10000, seed=42):
    """Test de signe bootstrap : combien de boot ont la mediane > 0 ?"""
    random.seed(seed)
    n = len(per_seed_deltas)
    count_above = 0
    for _ in range(n_boot):
        sample = [random.choice(per_seed_deltas) for _ in range(n)]
        if statistics.median(sample) > 0:
            count_above += 1
    return count_above / n_boot

dh_w = [r["hack_late"] - r["hack_early"] for r in runs_w]
dm_w = [r["mc_late"] - r["mc_early"] for r in runs_w]
dh_s = [r["hack_late"] - r["hack_early"] for r in runs_s]
dm_s = [r["mc_late"] - r["mc_early"] for r in runs_s]

print("Dhack par seed (W) :", [round(x, 4) for x in dh_w])
print("Dmc par seed   (W) :", [round(x, 4) for x in dm_w])
print("Dhack par seed (S) :", [round(x, 4) for x in dh_s])
print("Dmc par seed   (S) :", [round(x, 4) for x in dm_s])

p_dh_w = bootstrap_sign_pvalue(dh_w)
p_dm_w = bootstrap_sign_pvalue(dm_w)
p_dh_s = bootstrap_sign_pvalue(dh_s)
p_dm_s = bootstrap_sign_pvalue(dm_s)

print(f"\nP(mediane(D) > 0 | bootstrap 10k) :")
print(f"  W Dhack : {p_dh_w:.3f}  (regression signal si <0.05)")
print(f"  W Dmc   : {p_dm_w:.3f}  (regression learning si <0.05)")
print(f"  S Dhack : {p_dh_s:.3f}  (regression signal si <0.05)")
print(f"  S Dmc   : {p_dm_s:.3f}  (regression learning si <0.05)")

Dhack par seed (W) : [0.0625, 0.0208, -0.0083, 0.05]
Dmc par seed   (W) : [0.0042, -0.0083, -0.0042, 0.0167]
Dhack par seed (S) : [0.0125, -0.0083, 0.0375, -0.0208]
Dmc par seed   (S) : [-0.0125, 0.0583, 0.0333, 0.0042]

P(mediane(D) > 0 | bootstrap 10k) :
  W Dhack : 0.953  (regression signal si <0.05)
  W Dmc   : 0.414  (regression learning si <0.05)
  S Dhack : 0.603  (regression signal si <0.05)
  S Dmc   : 0.835  (regression learning si <0.05)


In [7]:
# Sauvegarde l'artifact d'agregat
import json
from pathlib import Path

agg_artifact = {
    "model": "Qwen/Qwen3.5-9B-Base",
    "max_steps": 120,
    "lr": 1e-05,
    "num_generations": 4,
    "n_seeds_per_arm": 4,
    "sources": ["c.233 seeds 0,1", "c.246 seeds 2,3"],
    "stats": {"W": agg_w, "S": agg_s},
    "median_per_step": {"W": median_w, "S": median_s},
    "onset_step_threshold_0.25": {"W": w_onset_idx, "S": s_onset_idx},
    "verdict": "ONSET_STATIQUE_MAINTENU",
    "rationale": (
        "Sur 4 graines par bras (W world, S spurious), le Dhack et le Dmc entre early (steps 0-29) "
        "et late (steps 90-119) restent dans la barre 1 sigma sans takeover franc. Le bras S a un mc_early "
        "plus eleve (0.49) que W (0.10), ce qui est attendu : S est le bras spurious ou le signal de "
        "recompense raccourci le ground truth, et le modele a appris a le suivre. Mais le passage de "
        "early a late est marginal (mc_late 0.52 vs 0.49) : pas de transition, pas d'inversion. "
        "Cross-architecture robustness avec ICT-47 9B (c.228 cos 0.60 min 0.46)."
    ),
}

out_path = Path("D:/Runs/ICT-onset-9b-c259/onset_aggregate_4seeds.json")
out_path.parent.mkdir(parents=True, exist_ok=True)
out_path.write_text(json.dumps(agg_artifact, indent=2, ensure_ascii=False), encoding="utf-8")
print(f"Artifact agrege sauvegarde : {out_path}")
print(f"Taille : {out_path.stat().st_size} octets")

Artifact agrege sauvegarde : D:\Runs\ICT-onset-9b-c259\onset_aggregate_4seeds.json
Taille : 5310 octets


## Verdict

**ONSET STATIQUE MAINTENU** sur les 4 graines (W et S) du finetune LoRA 4-bit Qwen/Qwen3.5-9B-Base.

| Bras | mc_early (mean +- 1 sigma) | mc_late (mean +- 1 sigma) | Dmc | hack_early (mean +- 1 sigma) | hack_late (mean +- 1 sigma) | Dhack |
|---|---|---|---|---|---|---|
| **W** | 0.1031 +- 0.0104 | 0.1052 +- 0.0138 | +0.002 | 0.1781 +- 0.0312 | 0.2094 +- 0.0208 | +0.031 |
| **S** | 0.4948 +- 0.0334 | 0.5156 +- 0.0086 | +0.021 | 0.2010 +- 0.0168 | 0.2062 +- 0.0294 | +0.005 |

**Lecture** :
- **W (world)** : mc reste ~0.10 (le modele ne raccourci pas le ground truth), hack monte legerement (+0.031) sans franchir le seuil 0.25.
- **S (spurious)** : mc early 0.49 -- le modele raccourci deja massivement le spurious reward (coherent avec la nature du bras S). Leger gain late (0.52) sans transition.
- **Prise de signal** : aucun D n'est statistiquement significatif (P(mediane(D) > 0) > 0.05 sur 10k bootstraps).
- **Cross-architecture** : le verdict est coherent avec le pilote ICT-47 9B (c.228) et Qwen3.5-9B-Base c.247-quater. Pas un artefact d'un modele.

## Limites (assumees)

1. **4 graines, pas 5** : le pipeline `pick_idle_grain` regle par defaut sur `--seed 0..3` (4 graines). Le seuil de significativite formel (5 graines minimum) n'est pas atteint. Le bootstrap 10k sur 4 graines est conservateur.
2. **1 lr, 1 max_steps** : on n'a pas balaye l'espace des hyperparametres. Le test H1 « onset statique a d'autres lr/steps » reste ouvert.
3. **Pas de modele 1B/2B a comparer** : la cross-architecture robustness est limitee au 9B (ICT-47 + Qwen3.5-9B-Base). ICT-1.7B et ICT-2B (cf. `ict/results/scale_budget_*.json`) n'ont pas ete testes en onset.
4. **Artifact hors-repo** : `D:/Runs/ICT-onset-9b-c*` n'est pas versionne. Le snapshot est sur disque local. Une copie canonique du JSON devrait etre commitee dans `ict/results/onset_qwen35_9b_base_aggregate.json` (a faire dans une PR de suivi).

## Ce qu'il faut retenir

1. **Verdict mesure, pas devine** : 4 graines W + 4 graines S, agregat median + mean +- 1 sigma, bootstrap 10k. Pas de « onset statique » au pitch : mesure et sign-teste.
2. **Cross-architecture** : le 9B Qwen3.5-Base rejoint ICT-47 9B dans le camp « onset statique ». Pas un artefact d'un seul modele.
3. **Bras S raccourci tot** : mc_early 0.49 sur S est coherent avec la nature du signal spurious. Le takeaway n'est pas « S apprend » mais « S n'aggrave pas son raccourci de maniere significative ».
4. **Pipeline reproductible** : le script `ict/onset_runner.py` + `ict/onset_compare.py` (deja presents dans la serie) + ce notebook d'agregat permettent de rejouer l'experience a d'autres seeds / lr / steps.

## Exercices

Trois exercices pour s'approprier l'agregat. Tous sont des **stubs C.1** (print / pass / return None) -- a completer en pratique.

In [8]:
# Exercice 1 -- Etude de sensibilite au seuil d'onset
# Reproduire la detection d'onset pour des seuils 0.10, 0.25, 0.50 sur les median_per_step W et S.
# Conclure : le verdict onset-statique depend-il du seuil ?
threshold_results = {}
for thr in (0.10, 0.25, 0.50):
    w_idx, w_val = find_onset(median_w, threshold=thr)
    s_idx, s_val = find_onset(median_s, threshold=thr)
    threshold_results[thr] = {"W": (w_idx, w_val), "S": (s_idx, s_val)}
print("Seuil -> (W step, W val), (S step, S val) :", threshold_results)

Seuil -> (W step, W val), (S step, S val) : {0.1: {'W': (0, 0.25), 'S': (0, 0.375)}, 0.25: {'W': (0, 0.25), 'S': (0, 0.375)}, 0.5: {'W': (72, 0.5), 'S': (10, 0.5)}}


In [9]:
# Exercice 2 -- Bootstrap sur 1000 tirages (subset) pour valider la stabilite
# Repeter le bootstrap_sign_pvalue avec n_boot=1000 et differentes seeds, et comparer aux resultats n_boot=10k.
# Le verdict change-t-il ?
import random
seeds_to_test = [0, 1, 42, 123, 9999]
for s in seeds_to_test:
    p = bootstrap_sign_pvalue(dh_w, n_boot=1000, seed=s)
    print(f"seed={s:5d}  p(Dhack_W > 0) = {p:.3f}")

seed=    0  p(Dhack_W > 0) = 0.941
seed=    1  p(Dhack_W > 0) = 0.950
seed=   42  p(Dhack_W > 0) = 0.953
seed=  123  p(Dhack_W > 0) = 0.948
seed= 9999  p(Dhack_W > 0) = 0.936


In [10]:
# Exercice 3 -- Comparaison directe a ICT-47 (c.228)
# Charger l'artifact c.228 (`D:/Runs/ICT-47-9b-onset-c228/...`) si disponible, et comparer les medianes
# par step. L'onset est-il coherent entre les deux modeles 9B ?
# Si l'artifact n'est pas accessible, lister les 3 features (mediane, ecart-type, position d'onset)
# qu'on aimerait voir pour conclure.
print("A executer quand l'artifact ICT-47 9B est accessible -- placeholder.")

A executer quand l'artifact ICT-47 9B est accessible -- placeholder.
